# Quarantine Manual Review

Records in `data/filtered/quarantined.parquet` are labeled **benign (`label=0`)** but triggered ModSecurity CRS heuristics during the quality gate (stage 3.4).  
Each record needs one of three decisions before it can rejoin the training set.

| Decision | Meaning | Outcome |
|---|---|---|
| **A** | Genuinely benign — CRS false positive | Restored to `filtered.parquet` as a hard negative |
| **B** | Mislabeled — CRS correctly caught it | Re-labeled `label=1`, written to `relabeled_malicious.parquet` |
| **D** | Ambiguous / unusable | Discarded entirely |

**Workflow:**
1. Run **Setup** and **Breakdown** to understand the batch
2. Run **Review records** — inspect each card; note `idx` values you want to override
3. Edit the `decisions` dict in **Decisions**
4. Run **Validate → Apply → Verify**

In [2]:
from pathlib import Path
import re
import html as _html
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from IPython.display import display, HTML

QUARANTINED = Path('../data/filtered/quarantined.parquet')
FILTERED    = Path('../data/filtered/filtered.parquet')

assert QUARANTINED.exists(), (
    f'Not found: {QUARANTINED}\n'
    'Run:  make data_augment_all  (completes through stage 3.4)'
)
qdf = pd.read_parquet(QUARANTINED)
print(f'Loaded {len(qdf)} quarantined records')
print(f'Columns: {list(qdf.columns)}\n')

if 'rejection_reason' not in qdf.columns:
    print(
        'WARNING: rejection_reason column not found.\n'
        'This file was generated before stage 3.4 added that column.\n'
        'Re-run  make data_augment_all  to get full CRS metadata.\n'
        'Continuing with rejection_reason=unknown -- breakdown charts will be empty.'
    )
    qdf['rejection_reason'] = 'unknown'

qdf[['id', 'source', 'attack_class', 'rejection_reason']]

Loaded 80925 quarantined records
Columns: ['id', 'method', 'path', 'query_string', 'headers', 'body', 'raw', 'label', 'attack_class', 'source', 'split']

This file was generated before stage 3.4 added that column.
Re-run  make data_augment_all  to get full CRS metadata.
Continuing with rejection_reason=unknown -- breakdown charts will be empty.


,id,source,attack_class,rejection_reason
0,93dd5a77-5531-40d7-8b20-ba7ea940f4b9,HTTPParams,sqli,unknown
1,6fab625d-5351-42c1-8f9a-3bfdf2c7834a,aug_synthesis_llm_path_traversal_crs_flagged,path_traversal,unknown
2,7c90ff4b-82c3-4f62-bb48-e611c62e85d2,aug_synthesis_llm_path_traversal,path_traversal,unknown
3,2559b968-de0f-48a9-98fd-691ae8bb8de3,HTTPParams,sqli,unknown
4,169d19ba-67b2-485f-83a3-c1b767ba1a29,HTTPParams,sqli,unknown
...,...,...,...,...
80920,1af2ca45-25c9-46df-a4b4-d05564dde257,aug_synthesis_grammar_path_traversal,path_traversal,unknown
80921,1c5e5374-3bcf-40e7-8eaa-c555da11db37,aug_benign_rest,benign,unknown
80922,560d10b8-24c1-4830-b2b3-1963464d3b41,aug_benign_rest,benign,unknown
80923,b7dca84a-6f70-4f30-a50c-7d40d768c49b,aug_synthesis_llm_path_traversal_crs_flagged,path_traversal,unknown


## Breakdown

In [ ]:
def _parse_crs(reason: str) -> list[str]:
    prefix = 'label_conflict:benign_with_crs_hit:'
    return reason.removeprefix(prefix).split(',') if reason.startswith(prefix) else []

qdf['crs_classes'] = qdf['rejection_reason'].apply(_parse_crs)

crs_counts = (
    qdf['crs_classes'].explode()
    .value_counts()
    .rename_axis('CRS class')
    .reset_index(name='records')
)
src_counts = (
    qdf['source'].value_counts()
    .rename_axis('source')
    .reset_index(name='records')
)

n_rows = max(len(crs_counts), len(src_counts), 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, max(3, 1 + n_rows * 0.4)))
if len(crs_counts):
    ax1.barh(crs_counts['CRS class'], crs_counts['records'], color='#4dabf7')
    ax1.invert_yaxis()
ax1.set_title('CRS patterns fired', fontweight='bold')
ax1.set_xlabel('records')

if len(src_counts):
    ax2.barh(src_counts['source'], src_counts['records'], color='#69db7c')
    ax2.invert_yaxis()
ax2.set_title('Source dataset', fontweight='bold')
ax2.set_xlabel('records')

plt.tight_layout()
plt.show()

print('CRS breakdown:')
display(crs_counts)
print('\nSource breakdown:')
display(src_counts)

## Review records

Each card shows the raw HTTP request with keyword matches highlighted in the colour of the CRS class that fired.  
Note the `idx` of any records you want to override in the **Decisions** cell.

In [ ]:
_CRS_COLORS = {
    'sqli': '#ff6b6b',
    'xss':  '#ffa94d',
    'lfi':  '#a9e34b',
    'ssrf': '#74c0fc',
    'cmdi': '#da77f2',
}

# Keyword lists for visual highlighting only -- not the authoritative filter logic
_CRS_KEYWORDS: dict[str, list[str]] = {
    'sqli': ['UNION SELECT', 'UNION ALL SELECT', 'SELECT', 'DROP TABLE',
             'INSERT INTO', 'OR 1=1', 'AND 1=1', 'SLEEP(', 'BENCHMARK(',
             'WAITFOR DELAY', 'information_schema', 'pg_sleep'],
    'xss':  ['<script', 'onerror=', 'onload=', 'javascript:', 'alert(',
             'document.cookie', '<iframe', '<svg'],
    'lfi':  ['../', 'php://', '/etc/passwd', '/etc/shadow'],
    'ssrf': ['169.254.169.254', 'metadata.google.internal',
             'gopher://', 'dict://', 'localhost/admin', '127.0.0.1/'],
    'cmdi': ['whoami', 'bash -i', 'nc -e', '/bin/sh', '/bin/bash'],
}


def _highlight_raw(text: str, crs_classes: list[str], max_len: int = 2_000) -> str:
    """HTML-escape text and wrap CRS keyword matches in coloured <mark> spans."""
    truncated = len(text) > max_len
    snippet = _html.escape(text[:max_len])
    for cls in crs_classes:
        color = _CRS_COLORS.get(cls, '#ffe066')
        for kw in _CRS_KEYWORDS.get(cls, []):
            pat = re.compile(re.escape(_html.escape(kw)), re.IGNORECASE)
            snippet = pat.sub(
                lambda m, c=color: (
                    '<mark style="background:' + c + ';color:#111;'
                    'border-radius:2px;padding:0 2px">' + m.group(0) + '</mark>'
                ),
                snippet,
            )
    if truncated:
        snippet += '<span style="color:#888"> [truncated]</span>'
    return snippet


def display_record(row) -> None:
    """Render one quarantine record as an HTML review card."""
    crs_classes: list[str] = row.get('crs_classes') or []
    badges = ' '.join(
        '<span style="background:' + _CRS_COLORS.get(c, '#ffe066') + ';color:#111;'
        'padding:2px 8px;border-radius:10px;font-size:11px;font-weight:bold">'
        + c.upper() + '</span>'
        for c in crs_classes
    )
    raw_html = _highlight_raw(str(row['raw']), crs_classes)
    header = (
        '<div style="background:#2d2d2d;color:#eee;padding:8px 14px;'
        'display:flex;gap:20px;flex-wrap:wrap;align-items:center">'
        f'<span><b>idx</b>&nbsp;{row.name}</span>'
        f'<span><b>id</b>&nbsp;{_html.escape(str(row["id"])[:8])}&#8230;</span>'
        f'<span><b>source</b>&nbsp;{_html.escape(str(row["source"]))}</span>'
        f'<span><b>class</b>&nbsp;{_html.escape(str(row["attack_class"]))}</span>'
        f'<span style="margin-left:auto">{badges}</span>'
        '</div>'
    )
    body = (
        '<pre style="margin:0;padding:10px 14px;background:#1a1a1a;color:#d4d4d4;'
        'white-space:pre-wrap;word-break:break-all;font-size:12px;'
        f'max-height:320px;overflow-y:auto">{raw_html}</pre>'
    )
    card = (
        '<div style="border:1px solid #444;border-radius:6px;margin:10px 0;'
        'font-family:monospace;overflow:hidden">'
        + header + body + '</div>'
    )
    display(HTML(card))

In [ ]:
for _, row in qdf.iterrows():
    display_record(row)

## Decisions

Edit the dict below.  
Keys are `idx` values from the card headers above; values are `"A"`, `"B"`, or `"D"`.

In [ ]:
# Default: restore everything as benign (CRS false positive).
# Override any record whose idx you noted while reviewing.

decisions: dict[int, str] = {idx: 'A' for idx in qdf.index}

# -- Overrides ----------------------------------------------------------------
# decisions[3]  = 'B'   # actually malicious
# decisions[7]  = 'D'   # too ambiguous to keep
# -----------------------------------------------------------------------------

summary = (
    pd.Series(decisions)
    .value_counts()
    .rename_axis('decision')
    .reset_index(name='count')
)
summary['meaning'] = summary['decision'].map({
    'A': 'restore benign',
    'B': 're-label malicious',
    'D': 'discard',
})
display(summary)

In [ ]:
bad_values = {k: v for k, v in decisions.items() if v not in {'A', 'B', 'D'}}
missing    = set(qdf.index) - set(decisions.keys())
extra      = set(decisions.keys()) - set(qdf.index)

assert not bad_values, f'Invalid decision values (must be A/B/D): {bad_values}'
assert not missing,    f'Missing decisions for indices: {missing}'
assert not extra,      f'Extra indices not in quarantine set: {extra}'

print(f'All {len(decisions)} decisions valid -- ready to apply.')

## Apply decisions

In [ ]:
from ai_waf_v2.data.schema import PARQUET_SCHEMA

schema_cols = [f.name for f in PARQUET_SCHEMA]
qdf['decision'] = pd.Series(decisions)

restore_df = qdf[qdf['decision'] == 'A'].copy()
relabel_df = qdf[qdf['decision'] == 'B'].copy()
discard_df = qdf[qdf['decision'] == 'D'].copy()

print(f'A -- restore  : {len(restore_df):>4} records')
print(f'B -- re-label : {len(relabel_df):>4} records')
print(f'D -- discard  : {len(discard_df):>4} records')

# -- A: append restored records to filtered.parquet --------------------------
if len(restore_df):
    restore_df['source'] = restore_df['source'] + '_reviewed_benign'
    existing  = pq.read_table(FILTERED).to_pandas()
    combined  = pd.concat([existing, restore_df[schema_cols]], ignore_index=True)
    pq.write_table(
        pa.Table.from_pandas(combined, schema=PARQUET_SCHEMA, preserve_index=False),
        FILTERED, compression='snappy',
    )
    print(f'\n  Restored {len(restore_df)} records \u2192 {FILTERED}')

# -- B: write re-labeled records separately ----------------------------------
if len(relabel_df):
    relabel_df['label']  = 1
    relabel_df['source'] = relabel_df['source'] + '_reviewed_malicious'
    out_path = FILTERED.parent / 'relabeled_malicious.parquet'
    pq.write_table(
        pa.Table.from_pandas(relabel_df[schema_cols], schema=PARQUET_SCHEMA, preserve_index=False),
        out_path, compression='snappy',
    )
    print(f'  Re-labeled {len(relabel_df)} records \u2192 {out_path}')
    print('  Next step: merge relabeled_malicious.parquet into filtered.parquet,')
    print('             then re-run stage 3.7 (stratified split).')

if len(discard_df):
    print(f'\n  Discarded {len(discard_df)} records (not written).')

print('\nDone.')

## Verify

In [ ]:
final      = pq.read_table(FILTERED).to_pandas()
n_restored = final['source'].str.contains('reviewed_benign', na=False).sum()

print(f'filtered.parquet  \u2192  {len(final):,} total records')
print(f'  label=0  benign    : {(final["label"] == 0).sum():,}')
print(f'  label=1  malicious : {(final["label"] == 1).sum():,}')
if n_restored:
    print(f'  from quarantine   : {n_restored} restored (source suffix _reviewed_benign)')